In [1]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. DATA PREPARATION
def prepare_dataset(json_data):
    prompts = []
    times = []
    lengths = []

    for program in json_data:
        for req in program['requests']:
            prompts.append(req['prompt'])
            times.append(req['kv_token_time'])
            # We add character length as a primary feature
            lengths.append(len(req['prompt']))

    # Convert text to TF-IDF features (content-based)
    vectorizer = TfidfVectorizer(max_features=500)
    X_tfidf = vectorizer.fit_transform(prompts).toarray()

    # Convert length to a 2D array and scale it
    X_len = np.array(lengths).reshape(-1, 1)
    len_scaler = StandardScaler()
    X_len_scaled = len_scaler.fit_transform(X_len)

    # Combine [TF-IDF Features (500) + Length Feature (1)] = 501 features
    X_combined = np.hstack((X_tfidf, X_len_scaled))

    # Scale the target variable (kv_token_time)
    y_scaler = StandardScaler()
    y = y_scaler.fit_transform(np.array(times).reshape(-1, 1))

    return X_combined, y, vectorizer, len_scaler, y_scaler

# 2. MODEL DEFINITION
class KVTimePredictor(nn.Module):
    def __init__(self, input_dim):
        super(KVTimePredictor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256), # Stabilizes training with mixed feature scales
            nn.ReLU(),
            nn.Dropout(0.2),     # Helps generalize with small datasets
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)    # Regression output
        )
        
    def forward(self, x):
        return self.network(x)

def train_model(X_train, y_train):
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.FloatTensor(y_train)
    
    model = KVTimePredictor(X_train_t.shape[1])
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("Starting training...")
    for epoch in range(101):
        model.train()
        optimizer.zero_grad()
        
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        
        loss.backward()
        optimizer.step()
        
        if epoch % 20 == 0:
            print(f"Epoch [{epoch}/100], Loss: {loss.item():.4f}")
    
    return model

# 4. INFERENCE FUNCTION
def predict_time(prompt, model, vectorizer, len_scaler, y_scaler):
    model.eval()
    with torch.no_grad():
        # Process input
        tfidf_feat = vectorizer.transform([prompt]).toarray()
        len_feat = len_scaler.transform([[len(prompt)]])
        
        # Combine and convert to tensor
        combined = np.hstack((tfidf_feat, len_feat))
        input_tensor = torch.FloatTensor(combined)
        
        # Predict
        scaled_prediction = model(input_tensor).item()
        
        # Reverse scaling to get actual kv_token_time
        actual_time = y_scaler.inverse_transform([[scaled_prediction]])[0][0]
        return actual_time


In [7]:
# Use json.load (no 's') to read from a file object
with open("kv_token_qwen_100token.json", "r") as f:
    raw_json = json.load(f)

# Now pass the loaded list/dict to your function
X, y, vec, l_scaler, y_scaler = prepare_dataset(raw_json)

# --- Dummy training setup for demonstration ---
# Assuming X.shape is (N, 501) and y.shape is (N, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y,train_size=0.1, test_size=0.1)

In [8]:
len(X_train), len(y_train), len(X_test), len(y_test)

(33394, 33394, 33395, 33395)

In [9]:
# Example Usage:
trained_model = train_model(X_train, y_train)

Starting training...
Epoch [0/100], Loss: 0.8327
Epoch [20/100], Loss: 0.0508
Epoch [40/100], Loss: 0.0391
Epoch [60/100], Loss: 0.0333
Epoch [80/100], Loss: 0.0286
Epoch [100/100], Loss: 0.0253


In [11]:
# To Save:
input_dim = X_train.shape[1]
bundle = {
    'model_state': trained_model.state_dict(),
    'vectorizer': vec,
    'len_scaler': l_scaler,
    'y_scaler': y_scaler,
    'input_dim': input_dim
}
torch.save(bundle, "full_predictor_package.pt")

# Load and use

In [15]:
# Load the whole dictionary
checkpoint = torch.load("full_predictor_package.pt", weights_only=False)

# Pull the dimension out of the dictionary
saved_dim = checkpoint['input_dim']

# Initialize the model with that specific dimension
loaded_model = KVTimePredictor(saved_dim)

# Now load the weights into that model
loaded_model.load_state_dict(checkpoint['model_state'])

<All keys matched successfully>

In [ ]:
# Load the whole dictionary
checkpoint = torch.load("full_predictor_package.pt", weights_only=False)

# Pull the dimension out of the dictionary
saved_dim = checkpoint['input_dim']

# Initialize the model with that specific dimension
loaded_model = KVTimePredictor(saved_dim)

# Now load the weights into that model
loaded_model.load_state_dict(checkpoint['model_state'])

import torch
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

# 1. Define the class (Required so PyTorch knows the architecture)
class KVTimePredictor(torch.nn.Module):
    def __init__(self, input_dim):
        super(KVTimePredictor, self).__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 256),
            torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(256, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 1)
        )
        
    def forward(self, x):
        return self.network(x)

# 2. Load the bundle
checkpoint = torch.load("full_predictor_package.pt", weights_only=False)

# 3. Reconstruct components
vec = checkpoint['vectorizer']
l_scaler = checkpoint['len_scaler']
y_scaler = checkpoint['y_scaler']
input_dim = checkpoint['input_dim']

model = KVTimePredictor(input_dim)
model.load_state_dict(checkpoint['model_state'])
model.eval() # Set to evaluation mode (turns off Dropout/BatchNorm updates)

# 4. PREDICTION FUNCTION
def predict_kv_time(prompt_text):
    with torch.no_grad():
        # A. Content features (TF-IDF)
        tfidf_feat = vec.transform([prompt_text]).toarray()
        
        # B. Length feature (Scaled)
        # We use the same scaler used during training
        raw_len = len(prompt_text)
        len_feat = l_scaler.transform([[raw_len]])
        
        # C. Combine [TF-IDF + Length]
        combined_input = np.hstack((tfidf_feat, len_feat))
        input_tensor = torch.FloatTensor(combined_input)
        
        # D. Model Forward Pass
        prediction_scaled = model(input_tensor)
        
        # E. Inverse Scale (Convert from normalized value back to ms/units)
        actual_time = y_scaler.inverse_transform(prediction_scaled.numpy())
        
        return actual_time[0][0]

# --- Example Usage ---
new_prompt = "Write a Python script to calculate the Fibonacci sequence up to 100."
estimated_time = predict_kv_time(new_prompt)

print(f"Predicted KV Token Time: {estimated_time:.2f}")

Predicted KV Token Time: 9076.32
